# Data Pipeline Runner
Runs full data pipeline on Kaggle: scrape Kenney + OGA → clean → caption → augment → push to HF.

In [ ]:
import os, sys, subprocess, json
from pathlib import Path

!pip install -q requests imagehash torchvision datasets huggingface_hub Pillow tqdm

if not os.path.exists('/kaggle/working/sprite-gen'):
    !git clone https://github.com/MANI8148/sprite-generator.git /kaggle/working/sprite-gen
sys.path.insert(0, '/kaggle/working/sprite-gen')

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")

print(f"HF_TOKEN set: {bool(HF_TOKEN)}")
if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN required. Add it as a Kaggle Secret.")

## Step 1: Scrape Kenney.nl

In [ ]:
!python /kaggle/working/sprite-gen/data/scripts/scrape_sources.py --output /kaggle/working/data/raw
print("Kenney scraping done")

## Step 2: Scrape OpenGameArt.org

In [ ]:
!python /kaggle/working/sprite-gen/data/scripts/scrape_oga.py --output /kaggle/working/data/raw --max-packs 50
print("OGA scraping done")

## Step 3: Scrape Itch.io (if URLs configured)

In [ ]:
!python /kaggle/working/sprite-gen/data/scripts/scrape_itchio.py --output /kaggle/working/data/raw
print("Itch.io scraping done")

## Step 4: Clean, normalize, dedup, palette

In [ ]:
!python /kaggle/working/sprite-gen/data/scripts/clean_normalize.py \
    --input /kaggle/working/data/raw --output /kaggle/working/data/processed \
    --canvas-size 32 --palette-size 24
print("Clean/normalize done")

## Step 5: AI Captioning (local heuristics)

In [ ]:
!python /kaggle/working/sprite-gen/data/scripts/caption_ai.py \
    --input /kaggle/working/data/processed \
    --output /kaggle/working/data/processed/metadata_labeled.json
print("Captioning done")

## Step 6: Augment dataset (4x)

In [ ]:
!python /kaggle/working/sprite-gen/data/scripts/augment_dataset.py \
    --input /kaggle/working/data/processed --output /kaggle/working/data/augmented \
    --copies 4
print("Augmentation done")

## Step 7: Push to HF Datasets

In [ ]:
!python /kaggle/working/sprite-gen/data/scripts/push_to_hf.py \
    --input /kaggle/working/data/augmented \
    --repo darklord8777/sprites \
    --token {HF_TOKEN}
print("Push done")

In [ ]:
import os
result = subprocess.run(['find', '/kaggle/working/data/augmented', '-name', '*.png'], capture_output=True, text=True)
png_count = len(result.stdout.strip().split('\n')) if result.stdout.strip() else 0
print(f"=== DATA PIPELINE COMPLETE ===")
print(f"Total PNGs on disk: {png_count}")